# DOTA + Strip R-CNN-S Notebook

This notebook is tailored to the DOTA-trained Strip R-CNN-S checkpoint in this repository. It gives you three workflows:

- load a pretrained Strip R-CNN-S detector
- render detections for only the classes you choose
- evaluate either the full annotated split or a selected subset of images

The notebook assumes the repository was installed with `pip install -e .` and that your DOTA split data follows the MMRotate directory layout.

## DOTA Class Note

DOTA does **not** contain literal `car` or `train` classes. Its closest vehicle labels are `small-vehicle` and `large-vehicle`, and it does contain `ship`.

If your practical goal is `car + ship`, use `small-vehicle`, `large-vehicle`, and `ship`. If you need a literal `train` detector, you will need a different dataset/checkpoint than the DOTA model used here.

In [ ]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mmcv
import numpy as np
import torch
from mmcv import Config
from mmcv.parallel import MMDataParallel
from mmdet.apis import init_detector, inference_detector, single_gpu_test
from mmdet.datasets import build_dataloader

import mmrotate  # noqa: F401
from mmrotate.apis import inference_detector_by_patches
from mmrotate.datasets import build_dataset


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'configs').exists() and (candidate / 'setup.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the Strip-R-CNN repository root from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f'Repository root: {REPO_ROOT}')
print(f'Torch version: {torch.__version__}')
print(f'MMRotate version: {mmrotate.__version__}')

In [ ]:
DOTA_CLASSES = (
    'plane',
    'baseball-diamond',
    'bridge',
    'ground-track-field',
    'small-vehicle',
    'large-vehicle',
    'ship',
    'tennis-court',
    'basketball-court',
    'storage-tank',
    'soccer-ball-field',
    'roundabout',
    'harbor',
    'swimming-pool',
    'helicopter',
)

CONFIG_PATH = REPO_ROOT / 'configs/strip_rcnn/strip_rcnn_s_fpn_1x_dota_le90.py'
CHECKPOINT_PATH = REPO_ROOT / 'checkpoints/strip_rcnn_s_dota_le90.pth'
DOTA_ROOT = REPO_ROOT / 'data/split_ms_dota'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

RENDER_CLASSES = ['ship', 'small-vehicle', 'large-vehicle']
VIS_SCORE_THR = 0.30
USE_PATCH_INFERENCE = False
PATCH_SIZES = [1024]
PATCH_STEPS = [824]
IMG_RATIOS = [1.0]
MERGE_IOU_THR = 0.10

SAMPLE_IMAGE = DOTA_ROOT / 'val/images/P0000.png'
MAX_VIS_IMAGES = 4

EVAL_SPLIT = 'val'
METRIC = 'mAP'
SAMPLES_PER_GPU = 1
WORKERS_PER_GPU = 2

SELECTED_IMAGE_IDS = []
MAX_SUBSET_IMAGES = None

print('Config path:', CONFIG_PATH)
print('Checkpoint path:', CHECKPOINT_PATH)
print('DOTA root:', DOTA_ROOT)
print('Device:', DEVICE)
print('Default render classes:', RENDER_CLASSES)

In [ ]:
def validate_render_classes(selected_classes, all_classes):
    selected = list(selected_classes or all_classes)
    unknown = [class_name for class_name in selected if class_name not in all_classes]
    if unknown:
        raise ValueError(f'Unknown class names: {unknown}. Valid DOTA classes are: {all_classes}')
    return selected


def configure_dota_paths(cfg: Config, dataset_root: Path) -> Config:
    cfg = cfg.copy()
    dataset_root = Path(dataset_root)

    if cfg.model.get('backbone') and cfg.model.backbone.get('init_cfg'):
        cfg.model.backbone.init_cfg = None
    if cfg.model.get('pretrained', None) is not None:
        cfg.model.pretrained = None

    cfg.data.train.ann_file = str(dataset_root / 'trainval/annfiles/')
    cfg.data.train.img_prefix = str(dataset_root / 'trainval/images/')

    cfg.data.val.ann_file = str(dataset_root / 'val/annfiles/')
    cfg.data.val.img_prefix = str(dataset_root / 'val/images/')

    cfg.data.test.ann_file = str(dataset_root / 'test/images/')
    cfg.data.test.img_prefix = str(dataset_root / 'test/images/')

    return cfg


def load_strip_rcnn_model(config_path: Path, checkpoint_path: Path, dataset_root: Path, device: str = DEVICE):
    config_path = Path(config_path)
    checkpoint_path = Path(checkpoint_path)
    dataset_root = Path(dataset_root)

    if not config_path.exists():
        raise FileNotFoundError(f'Config file not found: {config_path}')
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f'Checkpoint not found: {checkpoint_path}. Download the full DOTA detector checkpoint and update CHECKPOINT_PATH.'
        )
    if not dataset_root.exists():
        raise FileNotFoundError(
            f'DOTA root not found: {dataset_root}. Point DOTA_ROOT to the split DOTA directory.'
        )

    cfg = Config.fromfile(str(config_path))
    cfg = configure_dota_paths(cfg, dataset_root)
    model = init_detector(cfg, str(checkpoint_path), device=device)
    return model, cfg


def filter_result_by_classes(result, class_names, selected_classes):
    keep = set(validate_render_classes(selected_classes, class_names))
    filtered = []
    for class_name, class_result in zip(class_names, result):
        if class_name in keep:
            filtered.append(class_result)
        else:
            filtered.append(class_result[:0])
    return filtered


def run_detector(model, image_path, use_patch_inference=False):
    image_path = str(image_path)
    if use_patch_inference:
        return inference_detector_by_patches(
            model,
            image_path,
            sizes=PATCH_SIZES,
            steps=PATCH_STEPS,
            ratios=IMG_RATIOS,
            merge_iou_thr=MERGE_IOU_THR,
        )
    return inference_detector(model, image_path)


def visualize_detection(model, image_path, selected_classes=None, score_thr=VIS_SCORE_THR, use_patch_inference=USE_PATCH_INFERENCE, figsize=(10, 10)):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f'Image not found: {image_path}')

    selected_classes = validate_render_classes(selected_classes or RENDER_CLASSES, model.CLASSES)
    raw_result = run_detector(model, image_path, use_patch_inference=use_patch_inference)
    filtered_result = filter_result_by_classes(raw_result, model.CLASSES, selected_classes)

    rendered = model.show_result(
        str(image_path),
        filtered_result,
        score_thr=score_thr,
        show=False,
    )

    plt.figure(figsize=figsize)
    plt.imshow(mmcv.bgr2rgb(rendered))
    plt.title(f'{image_path.name} | rendered classes: {selected_classes}')
    plt.axis('off')
    plt.show()

    return raw_result, filtered_result


def visualize_image_list(model, image_paths, selected_classes=None, score_thr=VIS_SCORE_THR, use_patch_inference=USE_PATCH_INFERENCE):
    for image_path in image_paths:
        visualize_detection(
            model,
            image_path=image_path,
            selected_classes=selected_classes,
            score_thr=score_thr,
            use_patch_inference=use_patch_inference,
        )

In [ ]:
model, cfg = load_strip_rcnn_model(
    config_path=CONFIG_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    dataset_root=DOTA_ROOT,
    device=DEVICE,
)

print('Checkpoint classes:', model.CLASSES)

In [ ]:
raw_result, filtered_result = visualize_detection(
    model,
    image_path=SAMPLE_IMAGE,
    selected_classes=RENDER_CLASSES,
    score_thr=VIS_SCORE_THR,
    use_patch_inference=USE_PATCH_INFERENCE,
)

In [ ]:
candidate_images = sorted((DOTA_ROOT / 'val/images').glob('*.png'))[:MAX_VIS_IMAGES]
print([image_path.name for image_path in candidate_images])
visualize_image_list(
    model,
    image_paths=candidate_images,
    selected_classes=RENDER_CLASSES,
    score_thr=VIS_SCORE_THR,
    use_patch_inference=USE_PATCH_INFERENCE,
)

## Evaluation Helpers

For metric computation, use an annotated split such as `val` or `trainval`. The `test` split in the DOTA layout does not include labels, so it is appropriate for blind inference or submission formatting, not mAP calculation.

The subset evaluation helper below works by filtering the built dataset down to selected image ids before running the standard MMDetection single-GPU test loop.

In [ ]:
def build_dota_eval_dataset_cfg(cfg: Config, dataset_root: Path, split: str = 'val'):
    dataset_root = Path(dataset_root)
    split = split.lower()

    if split == 'val':
        dataset_cfg = copy.deepcopy(cfg.data.val)
        dataset_cfg['ann_file'] = str(dataset_root / 'val/annfiles/')
        dataset_cfg['img_prefix'] = str(dataset_root / 'val/images/')
    elif split == 'trainval':
        dataset_cfg = copy.deepcopy(cfg.data.val)
        dataset_cfg['ann_file'] = str(dataset_root / 'trainval/annfiles/')
        dataset_cfg['img_prefix'] = str(dataset_root / 'trainval/images/')
    elif split == 'test':
        dataset_cfg = copy.deepcopy(cfg.data.test)
        dataset_cfg['ann_file'] = str(dataset_root / 'test/images/')
        dataset_cfg['img_prefix'] = str(dataset_root / 'test/images/')
    else:
        raise ValueError("split must be one of: 'val', 'trainval', 'test'")

    dataset_cfg['test_mode'] = True
    return dataset_cfg


def subset_dataset(dataset, selected_image_ids=None, max_subset_images=None):
    selected_image_ids = [str(image_id).strip() for image_id in (selected_image_ids or []) if str(image_id).strip()]

    if not selected_image_ids and max_subset_images is None:
        return dataset

    selected_set = set(selected_image_ids)
    matched_indices = []
    matched_ids = []

    for index, info in enumerate(dataset.data_infos):
        image_id = Path(info['filename']).stem
        if not selected_set or image_id in selected_set:
            matched_indices.append(index)
            matched_ids.append(image_id)

    if selected_set:
        missing_ids = sorted(selected_set - set(matched_ids))
        if missing_ids:
            print('Warning: these image ids were not found in the selected split:', missing_ids)

    if max_subset_images is not None:
        matched_indices = matched_indices[:max_subset_images]

    if not matched_indices:
        raise ValueError('The selected subset is empty. Check SELECTED_IMAGE_IDS, MAX_SUBSET_IMAGES, and the chosen split.')

    subset = copy.deepcopy(dataset)
    subset.data_infos = [dataset.data_infos[index] for index in matched_indices]
    if hasattr(dataset, 'flag'):
        subset.flag = np.asarray(dataset.flag)[matched_indices]

    return subset


def evaluate_dota_split(
    model,
    cfg: Config,
    dataset_root: Path,
    split: str = 'val',
    metric: str | None = 'mAP',
    selected_image_ids=None,
    max_subset_images=None,
    samples_per_gpu: int = 1,
    workers_per_gpu: int = 2,
):
    split = split.lower()
    if split == 'test' and metric is not None:
        raise ValueError('DOTA test split has no labels in this layout. Use split=val or split=trainval for mAP evaluation.')

    dataset_cfg = build_dota_eval_dataset_cfg(cfg, dataset_root=dataset_root, split=split)
    dataset = build_dataset(dataset_cfg)
    dataset = subset_dataset(dataset, selected_image_ids=selected_image_ids, max_subset_images=max_subset_images)

    data_loader = build_dataloader(
        dataset,
        samples_per_gpu=samples_per_gpu,
        workers_per_gpu=workers_per_gpu,
        dist=False,
        shuffle=False,
    )

    base_model = model.module if hasattr(model, 'module') else model
    if next(base_model.parameters()).device.type != 'cuda':
        raise RuntimeError('This evaluation helper expects a CUDA device. For CPU-only evaluation, run tools/test.py directly.')

    device_index = next(base_model.parameters()).device.index
    if device_index is None:
        device_index = 0

    eval_model = MMDataParallel(base_model, device_ids=[device_index])
    outputs = single_gpu_test(eval_model, data_loader, show=False)

    metrics = None
    if metric is not None:
        metrics = dataset.evaluate(outputs, metric=metric)

    return dataset, outputs, metrics

In [ ]:
full_dataset, full_outputs, full_metrics = evaluate_dota_split(
    model=model,
    cfg=cfg,
    dataset_root=DOTA_ROOT,
    split=EVAL_SPLIT,
    metric=METRIC,
    selected_image_ids=None,
    max_subset_images=None,
    samples_per_gpu=SAMPLES_PER_GPU,
    workers_per_gpu=WORKERS_PER_GPU,
)

print(f'Images evaluated: {len(full_dataset)}')
full_metrics

In [ ]:
subset_dataset_obj, subset_outputs, subset_metrics = evaluate_dota_split(
    model=model,
    cfg=cfg,
    dataset_root=DOTA_ROOT,
    split=EVAL_SPLIT,
    metric=METRIC,
    selected_image_ids=SELECTED_IMAGE_IDS,
    max_subset_images=MAX_SUBSET_IMAGES,
    samples_per_gpu=SAMPLES_PER_GPU,
    workers_per_gpu=WORKERS_PER_GPU,
)

print(f'Images evaluated in subset: {len(subset_dataset_obj)}')
subset_metrics

## Recommended Usage

1. Edit `CHECKPOINT_PATH` and `DOTA_ROOT` first.
2. Set `RENDER_CLASSES` to the labels you want to draw.
3. Use the single-image cells for quick qualitative checks.
4. Use `EVAL_SPLIT='val'` or `'trainval'` for mAP.
5. Populate `SELECTED_IMAGE_IDS` or `MAX_SUBSET_IMAGES` when you want a subset-only evaluation run.